In [1]:
import pandas as pd

games = pd.read_csv("games.csv")
games_details = pd.read_csv("games_details.csv")
ranking = pd.read_csv("ranking.csv")

/var/folders/2r/46mxdql52bn78bp6pr2cps340000gn/T/ipykernel_6175/3465009782.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  games_details = pd.read_csv("games_details.csv")


In [3]:
games.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,2022-12-22,22200477,Final,1610612740,1610612759,2022,1610612740,126.0,0.484,0.926,...,25.0,46.0,1610612759,117.0,0.478,0.815,0.321,23.0,44.0,1
1,2022-12-22,22200478,Final,1610612762,1610612764,2022,1610612762,120.0,0.488,0.952,...,16.0,40.0,1610612764,112.0,0.561,0.765,0.333,20.0,37.0,1
2,2022-12-21,22200466,Final,1610612739,1610612749,2022,1610612739,114.0,0.482,0.786,...,22.0,37.0,1610612749,106.0,0.470,0.682,0.433,20.0,46.0,1
3,2022-12-21,22200467,Final,1610612755,1610612765,2022,1610612755,113.0,0.441,0.909,...,27.0,49.0,1610612765,93.0,0.392,0.735,0.261,15.0,46.0,1
4,2022-12-21,22200468,Final,1610612737,1610612741,2022,1610612737,108.0,0.429,1.000,...,22.0,47.0,1610612741,110.0,0.500,0.773,0.292,20.0,47.0,0


In [6]:
games['GAME_DATE_EST'] = pd.to_datetime(games['GAME_DATE_EST'])
games['GAME_DATE_EST']

0       2022-12-22
1       2022-12-22
2       2022-12-21
3       2022-12-21
4       2022-12-21
           ...    
26646   2014-10-06
26647   2014-10-06
26648   2014-10-06
26649   2014-10-05
26650   2014-10-04
Name: GAME_DATE_EST, Length: 26651, dtype: datetime64[ns]

In [8]:
games = games.sort_values('GAME_DATE_EST').reset_index(drop=True)

In [9]:
games['home_team_win'] = (games['PTS_home'] > games['PTS_away']).astype(int)

In [12]:
df = games[[
    'GAME_DATE_EST',
    'HOME_TEAM_ID',
    'VISITOR_TEAM_ID',
    'PTS_home',
    'PTS_away',
    'home_team_win'
]].copy()

In [13]:
df.isna().sum()

GAME_DATE_EST       0
HOME_TEAM_ID        0
VISITOR_TEAM_ID     0
PTS_home           99
PTS_away           99
home_team_win       0
dtype: int64

In [14]:
df = df.dropna().reset_index(drop=True)

In [15]:
df['point_diff'] = df['PTS_home'] - df['PTS_away']

In [16]:
home = df[['GAME_DATE_EST', 'HOME_TEAM_ID', 'home_team_win']].copy()
home.columns = ['date', 'team_id', 'win']

away = df[['GAME_DATE_EST', 'VISITOR_TEAM_ID', 'home_team_win']].copy()
away.columns = ['date', 'team_id', 'win']

# Flip win for away team
away['win'] = 1 - away['win']

team_games = pd.concat([home, away])
team_games = team_games.sort_values('date')

In [17]:
team_games['rolling_win_pct_5'] = (
    team_games
    .groupby('team_id')['win']
    .transform(lambda x: x.shift().rolling(5).mean())
)

In [18]:
df = df.merge(
    team_games[['date', 'team_id', 'rolling_win_pct_5']],
    left_on=['GAME_DATE_EST', 'HOME_TEAM_ID'],
    right_on=['date', 'team_id'],
    how='left'
).rename(columns={'rolling_win_pct_5': 'home_rolling_win_pct_5'})

df = df.merge(
    team_games[['date', 'team_id', 'rolling_win_pct_5']],
    left_on=['GAME_DATE_EST', 'VISITOR_TEAM_ID'],
    right_on=['date', 'team_id'],
    how='left'
).rename(columns={'rolling_win_pct_5': 'away_rolling_win_pct_5'})

In [19]:
df = df.drop(columns=['date_x', 'team_id_x', 'date_y', 'team_id_y'], errors='ignore')
df = df.dropna()

In [22]:
home = df[['GAME_DATE_EST', 'HOME_TEAM_ID', 'PTS_home', 'PTS_away']].copy()
home.columns = ['date', 'team_id', 'points_for', 'points_against']

away = df[['GAME_DATE_EST', 'VISITOR_TEAM_ID', 'PTS_away', 'PTS_home']].copy()
away.columns = ['date', 'team_id', 'points_for', 'points_against']

team_games = pd.concat([home, away]).sort_values('date')

In [23]:
team_games['point_diff'] = (
    team_games['points_for'] - team_games['points_against']
)

team_games['rolling_point_diff_5'] = (
    team_games
    .groupby('team_id')['point_diff']
    .transform(lambda x: x.shift().rolling(5).mean())
)

In [24]:
team_games['rolling_point_diff_5'] = (
    team_games
    .groupby('team_id')['point_diff']
    .transform(lambda x: x.shift().rolling(5).mean())
)

In [25]:
home = df[['GAME_DATE_EST', 'HOME_TEAM_ID', 'PTS_home', 'PTS_away']].copy()
home.columns = ['date', 'team_id', 'points_for', 'points_against']

away = df[['GAME_DATE_EST', 'VISITOR_TEAM_ID', 'PTS_away', 'PTS_home']].copy()
away.columns = ['date', 'team_id', 'points_for', 'points_against']

team_games = pd.concat([home, away]).sort_values('date')

In [26]:
team_games = team_games.sort_values(['team_id', 'date'])

team_games['prev_game_date'] = (
    team_games.groupby('team_id')['date'].shift()
)

team_games['rest_days'] = (
    team_games['date'] - team_games['prev_game_date']
).dt.days

In [27]:
df = df.merge(
    team_games[['date', 'team_id', 'rest_days']],
    left_on=['GAME_DATE_EST', 'HOME_TEAM_ID'],
    right_on=['date', 'team_id'],
    how='left'
).rename(columns={'rest_days': 'home_rest_days'})

df = df.merge(
    team_games[['date', 'team_id', 'rest_days']],
    left_on=['GAME_DATE_EST', 'VISITOR_TEAM_ID'],
    right_on=['date', 'team_id'],
    how='left'
).rename(columns={'rest_days': 'away_rest_days'})

In [28]:
df = df.drop(columns=['date_x', 'team_id_x', 'date_y', 'team_id_y'], errors='ignore')
df = df.dropna()